# Building a GNN + PPO Agent for Orbit Wars

**A complete walkthrough — from graph representation to reinforcement learning**

This notebook explains how to build a competitive agent for the [Orbit Wars](https://www.kaggle.com/competitions/orbit-wars) Kaggle competition using a **Graph Neural Network** (GNN) trained with **Proximal Policy Optimisation** (PPO).

We cover:
1. Why graphs are the right representation for this game
2. Designing node features, edge features, and message passing
3. The Graph Attention Network (GAT) architecture with edge-conditioned messages
4. Factored action spaces (source → target → fraction)
5. Behavioral Cloning (BC) pre-training from Kaggle replays
6. PPO fine-tuning with curriculum opponents and reward shaping
7. Separate actor/critic backbones and why they matter
8. Progressive horizon training
9. All hyperparameters explained

---

## 1. The Game: Orbit Wars

Orbit Wars is a real-time strategy game played on a 100×100 board with a **sun at the centre** (position 50, 50). Planets orbit the sun, and each planet has:
- An **owner** (you, an enemy, or neutral)
- A **ship garrison** (military units)
- A **production rate** (ships produced per step)
- An **orbital velocity** (how fast it moves around the sun)

Each turn, you can launch fleets from your planets to attack or reinforce other planets. Fleets travel in straight lines at a fixed speed (6 units/step). **Fleets that cross the sun are destroyed**, so routing matters.

Games last up to 500 steps. In **2-player** mode, the last player standing wins. In **4-player** mode, the dynamics are far more complex — alliances shift, you can be attacked from multiple directions, and overly aggressive play gets you eliminated.

### Why This Is Hard

- The number of planets varies per game (typically 15–45)
- Planets **move** — their positions change every step as they orbit
- You must decide **which** planet to launch from, **where** to send ships, and **how many** to send
- The action space is enormous: every owned planet × every target planet × fraction of ships
- You need different strategies for 2p and 4p modes

## 2. Why Graphs?

The game state is naturally a **graph**:
- **Nodes** = planets (variable number per game)
- **Edges** = potential fleet paths between every pair of planets

Standard neural networks like CNNs or MLPs struggle here because:
- The number of planets changes between games (variable-size input)
- Spatial relationships between planets are what matter, not pixel positions
- We need **permutation invariance** — renumbering the planets shouldn't change the policy

A **Graph Neural Network** (GNN) handles all of these naturally. It processes each node using information from its neighbors, producing per-node embeddings that capture both local and global context.

```
Planet Graph (example with 5 planets):

    [P0: mine, 50 ships]  ←--edge features--→  [P1: enemy, 30 ships]
         |                                            |
    edge features                                edge features
         |                                            |
    [P2: mine, 20 ships]  ←--edge features--→  [P3: neutral, 10 ships]
                    \                               /
                     ←--edge features--→  [P4: enemy, 40 ships]
```

Every planet is connected to every other planet (fully connected graph), because you can potentially send a fleet anywhere. The **edge features** encode the spatial relationship between each pair.

## 3. Feature Engineering

### 3.1 Node Features (28 dimensions)

Each planet gets a 28-dimensional feature vector. These are carefully designed to give the network all the information it needs to make good decisions:

| Index | Feature | Normalisation | Why It Matters |
|-------|---------|---------------|----------------|
| 0-1 | Position (x, y) | ÷ 100 | Spatial awareness, distance to sun |
| 2 | Is mine | 0/1 | Ownership — fundamental to all decisions |
| 3 | Is enemy | 0/1 | Know what to attack |
| 4 | Is neutral | 0/1 | Know what to expand to |
| 5 | Ship count | log1p ÷ 6.5 | Military strength (log-scaled to handle huge values) |
| 6 | Production rate | ÷ 10 | Economic value — high-production planets are strategic targets |
| 7 | Threat ratio | clamp(inbound\_enemy / ships, 0, 3) ÷ 3 | Am I about to be attacked? |
| 8 | Inbound friendly ships | log1p ÷ 6.5 | Reinforcements incoming |
| 9 | Inbound enemy ships | log1p ÷ 6.5 | Threats incoming |
| 10 | Step fraction | step ÷ max\_steps | Game phase — early vs late game needs different strategies |
| 11 | Remaining production value | prod × remaining\_frac × 2 | How valuable is this planet for the rest of the game? |
| 12 | Production advantage | clamp((my\_prod − enemy\_prod) ÷ 20, −1, 1) | Am I winning or losing economically? |
| 13 | Nearest enemy distance | ÷ 100 | How exposed is this planet? |
| 14 | Nearest enemy fleet ETA | clamp(eta ÷ 50, 0, 1) | How urgently do I need to defend? |
| 15 | Ship surplus ratio | clamp((surplus − threat) ÷ 6.5, −1, 1) | Can I afford to send ships away? |
| 16 | Orbital angular velocity | ÷ 4 | Fast-orbiting planets change neighbours quickly |
| 17 | Number of players | num\_players ÷ 4 | 2p vs 4p needs fundamentally different strategies |
| 18 | My planet fraction | my\_planets ÷ total\_owned | Am I dominant or underdog? |
| 19 | My military share | my\_total\_ships ÷ total\_ships | Global military balance |
| 20 | Strongest enemy share | strongest\_enemy\_ships ÷ total\_ships | Who's the biggest threat? |
| 21 | Friendly neighbors | count within 30 units ÷ 5 | Local support network |
| 22 | Enemy neighbors | count within 30 units ÷ 5 | Local threat density |
| 23 | Local force balance | (friendly − enemy ships nearby) ÷ total, clamp(−1, 1) | Am I strong or weak here? |
| 24 | Is frontline | 1 if nearest enemy < nearest friendly | Should this planet attack or defend? |
| 25 | Inbound friendly fleet count | count ÷ 5 | How many separate reinforcement waves? |
| 26 | Inbound enemy fleet count | count ÷ 5 | How many separate attack waves? |
| 27 | In-transit ship ratio | my\_fleet\_ships ÷ my\_total\_ships | How much of my army is airborne? |

#### Design Principles

- **All features are normalised to roughly [0, 1] or [−1, 1]** — this prevents any single feature from dominating gradient updates
- **Log-scaling for ship counts** — ship counts can range from 0 to 1000+; `log1p` compresses this range
- **Global context features** (17–20, 27) — give the network a sense of the overall game state, not just local information
- **Local neighborhood features** (21–24) — capture spatial clustering without needing the GNN to learn it from scratch
- **Temporal features** (10, 11) — the optimal strategy changes dramatically between early game (expand) and late game (attack)

We started with just 10 features (the basics: position, ownership, ships, production) and expanded to 16, then 28. Each expansion improved learning speed because the network has to discover less on its own.

### 3.2 Edge Features (6 dimensions)

Every pair of planets has a 6-dimensional edge feature vector:

| Index | Feature | Why It Matters |
|-------|---------|----------------|
| 0 | Distance (normalised) | How far away is this target? |
| 1 | Travel time | How many steps will the fleet take? (distance ÷ ship\_speed) |
| 2 | Angle sine | Direction of travel (sin component) |
| 3 | Angle cosine | Direction of travel (cos component) |
| 4 | Sun intersects | **Binary**: does the straight-line path cross the sun's danger zone? |
| 5 | Sun clearance | How close does the path get to the sun? (normalised) |

The **sun features** (4–5) are critical. In Orbit Wars, fleets that cross the sun are destroyed. The network must learn to avoid these paths. By encoding this directly as an edge feature, we make it easy — the attention mechanism can learn "never send ships along sun-blocked edges".

We also use the sun intersection as a **hard mask** during action selection: if a path crosses the sun, we set that target's logit to −∞, making it impossible to select. This gives us a safety guarantee that no training can override.

```python
# Sun geometry computation
def closest_approach_to_sun(x1, y1, x2, y2):
    """Closest distance from the path (x1,y1)->(x2,y2) to the sun centre."""
    # Project sun onto the line segment, clamp to [0,1]
    t = clamp(dot(sun - p1, p2 - p1) / |p2 - p1|², 0, 1)
    return distance(p1 + t * (p2 - p1), sun)
    
sun_radius = 10.0  # danger zone
safety_margin = 2.0  # extra buffer
```

## 4. The GNN Architecture: GAT with Edge-Conditioned Messages

We use a **Graph Attention Network** (GAT) — a type of GNN where each node pays different amounts of "attention" to its neighbors. This is powerful because not all planets are equally relevant: your frontline planet should pay most attention to nearby enemy planets, not distant friendly ones.

### 4.1 Standard GAT vs Our Enhancement

In a **standard GAT**, edge features only affect the **attention weights** (how much to listen to each neighbor). The actual message from a neighbor is just its node embedding.

Our enhancement: edge features appear in **both** the attention weights **AND** the message values. This means the network can learn things like:

> *"Planet j has 50 ships AND is only 3 steps away — this is a critical threat"*

With standard GAT, the network knows planet j has 50 ships and that it's close (via attention), but the **message** from j doesn't encode "closeness". With our edge-in-message approach, the message itself carries spatial information.

### 4.2 How a Single GAT Layer Works

```
Input: node embeddings h (B, N, D), edge features e (B, N, N, edge_dim)

Step 1: Project nodes
    Wh = W · h                    # (B, N, H, D/H) — H heads, D/H dims each

Step 2: Compute attention scores
    score = a_src · Wh_i           # how important is the source?
          + a_tgt · Wh_j           # how important is the target?
          + a_edge · encode(e_ij)  # how important is the spatial relationship?
    
    α_ij = softmax_j(LeakyReLU(score))   # normalise over all neighbors j

Step 3: Compute messages (THIS IS OUR KEY ENHANCEMENT)
    msg_j→i = Wh_j + edge_val(e_ij)   # neighbor embedding + edge embedding
                                        # Standard GAT would just use Wh_j here

Step 4: Aggregate
    h'_i = Σ_j  α_ij · msg_j→i        # attention-weighted sum of messages

Step 5: Residual + LayerNorm
    output = LayerNorm(h'_i + residual(h_i))
```

### 4.3 Multi-Head Attention

We use **4 attention heads**, each with `hidden_dim / 4` dimensions. This lets the network learn different types of relationships in parallel:
- Head 1 might focus on nearby threats
- Head 2 might focus on production-rich targets
- Head 3 might focus on friendly reinforcement paths
- Head 4 might focus on sun-safe routes

The outputs are concatenated back to `hidden_dim` dimensions.

### 4.4 The Full Network Architecture

```
                    ┌─────────────────────────────────────────┐
                    │         POLICY BACKBONE (Actor)         │
                    │                                         │
Node Features ──→  │  Node Encoder (28 → 256 → 256, ReLU)    │
  (B, N, 28)       │         ↓                                │
                    │  Edge Encoder (6 → 128 → 128, ReLU)     │ ←── Edge Features
                    │         ↓                                │      (B, N, N, 6)
                    │  GAT Layer 1 (256→256, 4 heads)         │
                    │         ↓                                │
                    │  GAT Layer 2 (256→256, 4 heads)         │
                    │         ↓                                │
                    │  Global Pool (mean) → Project (256)     │
                    │         ↓                                │
                    │  Node embeddings h (B,N,256)             │
                    │  Global context g (B, 256)               │
                    └────┬────────┬─────────┬─────────────────┘
                         │        │         │
                    ┌────▼──┐ ┌───▼───┐ ┌───▼────┐
                    │Source │ │Target │ │Fraction│
                    │ Head  │ │ Head  │ │ Head   │
                    └───────┘ └───────┘ └────────┘

                    ┌─────────────────────────────────────────┐
                    │        CRITIC BACKBONE (separate!)      │
                    │                                         │
Same inputs ──→     │  Critic Node Encoder (28→256→256)       │
                    │  Critic Edge Encoder (6→128→128)        │
                    │  Critic GAT Layer 1                     │
                    │  Critic GAT Layer 2                     │
                    │  Critic Global Pool → Project           │
                    │         ↓                                │
                    │    Value Head (256→256→1)                │
                    └─────────────────────────────────────────┘
```

**Total parameters: ~1.24M** (883K policy + 357K critic backbone)

The key point: the **policy** and **critic** have completely separate GNN backbones. We'll explain why in Section 8.

## 5. Factored Action Space

The action space is decomposed into three sequential decisions:

### Step 1: Source Selection
"Which planet do I launch from?" (or do nothing)

- One logit per planet + one **noop** logit
- **Masked**: only owned planets with ships > 0 can be sources
- Noop bias: initialised to −2.0 so the agent prefers acting early in training

### Step 2: Target Selection (conditioned on source)
"Where do I send the fleet?"

- Logits computed from: `[source_embed, target_embed, edge_features, global_context]`
- **Masked**: can't target yourself; can't target through the sun
- The edge features here are crucial — they encode distance, travel time, and sun safety

### Step 3: Fraction Selection (conditioned on source + target)
"How many ships do I send?"

- 4 discrete buckets: **25%, 50%, 75%, 100%**
- Logits from: `[source_embed, target_embed, global_context]`

### Joint Log-Probability

$$\log p(\text{action}) = \log p(\text{source}) + \log p(\text{target} | \text{source}) + \log p(\text{fraction} | \text{source, target})$$

When **noop** is selected, target and fraction log-probs are 0 (they don't apply).

This factored approach reduces the action space dramatically. Instead of `N × N × 4` flat actions (potentially thousands), we make three smaller decisions with `N+1`, `N`, and `4` options respectively.

## 6. Phase 1: Behavioral Cloning (BC)

Before doing any reinforcement learning, we **pre-train** the network to imitate winning players from Kaggle replays. This is called **Behavioral Cloning**.

### Why Pre-Train?

PPO starts from random weights. A random policy in Orbit Wars does essentially nothing useful — it sends tiny random fleets in random directions. Learning from scratch would take an extremely long time because the reward signal is so sparse (you only win or lose after 500 steps).

BC gives the agent a "warm start" — it already knows roughly how to play before PPO begins. Think of it as teaching someone the rules and basic strategy before they start practising.

### The BC Pipeline

```
Kaggle Replays (JSON) → Parse Actions → Build Graphs → Train
    │                        │               │           │
    │  ~3,500 replays        │ angle → target │ 28-dim    │ Cross-entropy
    │  winners only          │ matching       │ features  │ loss
    └────────────────────────┘               └───────────┘
```

#### Replay Parsing

Kaggle replays store actions as `[source_planet_id, angle_in_radians, num_ships]`. We need to convert these to our factored representation:

1. **Angle → Target**: We trace a ray from the source planet at the given angle and find the closest planet within a cone tolerance (0.3 radians)
2. **Ships → Fraction**: We find the closest bucket (25%, 50%, 75%, 100%) to `ships_sent / ships_available`
3. **No action → Noop**: If a player takes no action on a step, that's a noop

We only learn from **winning players** — this gives the network a curriculum of good play.

#### BC Loss Function

```python
loss = CE(source_logits, source_label) 
     + CE(target_logits, target_label)   # only for non-noop actions
     + CE(fraction_logits, fraction_label)  # only for non-noop actions
```

Where CE is cross-entropy loss. For noop actions, only the source component applies.

#### BC Training Details

```python
# BC hyperparameters
hidden_dim = 256        # GNN hidden dimension
batch_size = 512        # samples per batch
learning_rate = 3e-4    # Adam optimizer
epochs = 10-20          # until validation loss plateaus
train/val split = 90/10
device = 'mps'          # Apple Silicon GPU (or 'cuda')
```

We train until the validation loss stops improving (typically around val\_loss ≈ 3.71).

## 7. Phase 2: PPO Fine-Tuning

Behavioral Cloning can only be as good as the replays it learns from. To go further, we use **Proximal Policy Optimisation** (PPO) — the agent plays actual games and improves from the outcomes.

### 7.1 What is PPO?

PPO is a policy gradient method that updates the policy to increase the probability of actions that led to good outcomes and decrease the probability of actions that led to bad outcomes. The key innovation of PPO is the **clipped objective**:

$$L^{CLIP} = \mathbb{E}\left[\min\left(r_t \hat{A}_t, \text{clip}(r_t, 1-\epsilon, 1+\epsilon) \hat{A}_t\right)\right]$$

Where:
- $r_t = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{old}}(a_t|s_t)}$ is the probability ratio (how much has the policy changed?)
- $\hat{A}_t$ is the advantage estimate (was this action better or worse than expected?)
- $\epsilon = 0.2$ is the clip range

The clipping prevents the policy from changing too drastically in a single update, which makes training stable.

### 7.2 Advantage Estimation (GAE)

We use **Generalised Advantage Estimation** (GAE) to compute how good each action was:

$$\hat{A}_t = \sum_{l=0}^{\infty} (\gamma \lambda)^l \delta_{t+l}$$

Where $\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$ is the TD error.

- **γ (gamma) = 0.997**: Discount factor. High value because Orbit Wars is a long game — actions early on affect the outcome 400+ steps later
- **λ (lambda) = 0.95**: GAE smoothing parameter. Balances bias vs variance in advantage estimates

### 7.3 Reward Shaping

The raw game reward is just win/loss at the end. This is too sparse for effective learning. We add **dense reward shaping** to give the agent feedback every step:

#### Per-Step Dense Reward

```python
# Score = fleet + 3× production (production is worth more because it compounds)
score = (my_fleet + 3.0 * my_prod) - (enemy_fleet + 3.0 * enemy_prod)
delta = score - previous_score  # how much did I improve this step?

# Also penalise enemy improvement (zero-sum thinking)
enemy_delta = enemy_total - previous_enemy_total

shaped_reward = delta * 0.01 - enemy_delta * 0.005 - step_penalty

# Idle penalty: punish sitting still when you have ships and planets
if is_noop and has_planets and my_fleet > 50:
    shaped_reward -= idle_penalty  # 0.02
```

#### Terminal Reward

```python
win_bonus = 10.0  # dominates the dense shaping (which uses 0.01 scale)

if won:       final = +10.0
if lost:      final = -10.0 * (1 - 0.5 * survival_fraction)  
              # dying late is less bad than dying early
if timeout:   # game hit horizon limit
              # differential metric based on fleet/production advantage
              advantage = (1-time_value) * fleet_diff + time_value * 1.5 * prod_diff
              final = tanh(advantage) * 5.0
```

The **tanh** on the timeout advantage prevents extreme reward values from destabilising training.

#### Noop Penalty (Logit Bias)

Separately from reward shaping, we apply a **logit penalty** to the noop action during inference:

```python
source_logits[noop_index] -= noop_penalty  # default: 4.0
```

This makes the agent strongly prefer taking actions over doing nothing. Without this, newly initialised policies tend to learn that "doing nothing" is safe (no negative reward) and get stuck.

### 7.4 PPO Update Mechanics

```
Collect 16 episodes (rollouts)
    ↓
Compute GAE advantages + returns
    ↓
Normalise advantages (mean=0, std=1)
    ↓
For 4 epochs:
    Shuffle data, split into mini-batches (size 64)
    For each mini-batch:
        - Compute new log_prob, entropy, value
        - ratio = exp(new_log_prob - old_log_prob)
        - Clipped policy loss
        - Value loss (MSE)
        - Entropy bonus
        - Total loss = policy + 0.5 * value - 0.05 * entropy
        - Gradient step
```

Key details:
- **episodes\_per\_update = 16**: We collect 16 full game episodes before doing a PPO update. More data per update = more stable gradients
- **mini\_batch\_size = 64**: Each update epoch processes the rollout in chunks of 64 timesteps
- **4 PPO epochs**: We reuse each batch of experience 4 times (standard for PPO)
- **Devices**: Rollouts run on CPU (game environment isn't GPU-compatible), PPO updates run on MPS/CUDA for speed

## 8. Separate Actor and Critic — Why It Matters

This is one of the most important architectural decisions. In standard actor-critic methods, the **policy** (actor) and **value function** (critic) share the same neural network backbone:

```
SHARED BACKBONE (standard):           SEPARATE BACKBONES (ours):

Input → GNN → Embeddings              Input → Policy GNN → Policy Embeddings
                ├→ Policy heads                                  ├→ Source head
                └→ Value head                                    ├→ Target head
                                                                 └→ Fraction head
                                       Input → Critic GNN → Critic Embeddings
                                                                 └→ Value head
```

### The Problem with Sharing

The policy and value function have **conflicting objectives**:

- The **policy** needs to learn "which action is best?" — it needs sharp, discriminative features
- The **value function** needs to learn "how good is this state?" — it needs smooth, predictive features

When they share a backbone, gradients from the value loss can **corrupt** the policy's feature representations and vice versa. This is especially problematic during **self-play**, where the opponent is a copy of yourself. As the value function adjusts to the changing self-play opponent, it drags the policy features around, causing instability.

### The Solution

We give the critic its own complete GNN backbone — separate `node_encoder`, `edge_encoder`, `gnn1`, `gnn2`, and `global_proj`. The only shared component is the input (same node/edge features).

This adds ~40% more parameters (883K → 1.24M), but the training is much more stable. The policy can evolve its representations freely without worrying about breaking the value function.

### Warm-Starting the Critic

When loading a BC checkpoint (which only has the shared backbone), we **copy** the policy backbone weights into the critic backbone:

```python
# Initialise critic from policy weights
copy_map = {
    'node_encoder': 'critic_node_encoder',
    'edge_encoder': 'critic_edge_encoder', 
    'gnn1': 'critic_gnn1',
    'gnn2': 'critic_gnn2',
    'global_proj': 'critic_global_proj',
}
```

This gives the critic a good starting point — it already understands the game state — but from this point on, the two backbones diverge as they're updated by different loss functions.

## 9. Curriculum Training

We don't throw the agent against the strongest opponents immediately. Instead, we use a **curriculum** — starting with easy opponents and gradually increasing difficulty as the agent improves.

### 9.1 The 8-Stage Curriculum

| Stage | Name | Max Tier | Advance WR | Self-Play % | Opponents |
|-------|------|----------|------------|-------------|----------|
| 0 | Floor | 1 | 65% | 0% | Random only |
| 1 | Beginner | 2 | 65% | 5% | + Starter, Bully, Prospector |
| 2 | Novice | 3 | 58% | 10% | + Nearest Sniper, Rage, Dual |
| 3 | Developing | 4 | 52% | 15% | + Baseline, Sig Starter |
| 4 | Intermediate | 5 | 46% | 20% | + Pascal v14, Ykhnkf |
| 5 | Competent | 6 | 40% | 25% | + Shunlite, v131, Sig Reinforce |
| 6 | Advanced | 7 | 35% | 30% | + v131 Denial, v131 Wave, Tamrazov |
| 7 | Elite | 8 | — | 35% | + Pilkwang (terminal stage) |

### How Advancement Works

The agent advances to the next stage when its **heuristic win rate** (against the tiered opponents, excluding self-play) exceeds the stage's `advance_wr` threshold over a minimum number of games.

Win rate thresholds decrease as stages increase because the opponents get harder — maintaining 65% against random is easy, but 35% against tier-7 bots is impressive.

### 9.2 Self-Play

From stage 1 onwards, a fraction of games are played against a **frozen copy of itself** (self-play). This copy is refreshed every 20 episodes. Self-play teaches the agent to beat strategies it has already learned, pushing it beyond imitation.

Self-play starts at just 5% and ramps to 35%. Starting too high too early is counterproductive — a random policy playing against itself learns nothing.

### 9.3 Opponent Sampling

Within the available tier range, opponents are sampled with **tier-weighted probability**:

```python
weight = (max_tier - tier + 1) ** 2
```

This heavily favours easier opponents, ensuring the agent always has some "winnable" games for stable gradient signals, even at advanced stages. Without this, the agent can get stuck losing every game and learning nothing.

## 10. Progressive Horizon Training

Full Orbit Wars games last 500 steps. Training on 500-step games from the start is problematic:
- Credit assignment is extremely difficult over 500 steps
- Each episode takes a long time (slow iteration)
- The agent needs to learn basic mechanics before strategic long-term planning

### Performance-Gated Horizons

We start with short games and extend as the agent proves competence:

| Heuristic WR Threshold | Max Steps | Rationale |
|----------------------|-----------|----------|
| 0% (start) | 100 | Learn basic fleet launching and planet capture |
| 50% | 200 | Extend to mid-game territory control |
| 60% | 350 | Learn production accumulation and timing |
| 70% | 500 | Full game — endgame strategies and elimination |

The horizon only advances when the agent achieves the required win rate at the current horizon. There's no time pressure — it stays at each level until it's genuinely ready.

This is similar to how humans learn chess: first learn piece movement (short puzzles), then tactics (mid-game), then full-game strategy.

## 11. Mixed 2P/4P Training

The agent needs to play both 2-player and 4-player games on Kaggle. These modes require fundamentally different strategies:

- **2P**: Direct aggression, binary win/loss, focus on one opponent
- **4P**: Diplomacy matters, avoid being the weakest (get eliminated first) or strongest (everyone attacks you), manage multiple fronts

We train on **mixed** games, starting with 95% 2p / 5% 4p and gradually shifting to 50/50 over 10,000 episodes:

```python
# 4p fraction ramps linearly from 5% to 50%
frac_4p = 0.05 + (0.50 - 0.05) * min(episode / 10000, 1.0)
```

The agent receives the **number of players** as a node feature (feature [17]), so it can learn mode-specific strategies with the same network.

In 4p games, the agent plays as one of the four players. The other three are drawn from the opponent pool. Win/loss is determined by final ranking.

## 12. Complete Hyperparameter Reference

### Network Architecture
| Parameter | Value | Notes |
|-----------|-------|-------|
| Node feature dim | 28 | Expanded from original 10 |
| Edge feature dim | 6 | Distance, travel time, angle, sun |
| Hidden dim | 256 | GNN embedding size |
| GAT heads | 4 | Multi-head attention |
| GAT layers | 2 | Depth of message passing |
| Fraction buckets | [0.25, 0.5, 0.75, 1.0] | Discrete ship fractions |
| Separate critic | Yes | Independent GNN for value function |
| Total params | ~1.24M | 883K policy + 357K critic |

### BC Pre-Training
| Parameter | Value | Notes |
|-----------|-------|-------|
| Learning rate | 3e-4 | Adam optimizer |
| Batch size | 512 | |
| Epochs | 10-20 | Until val loss plateaus |
| Dataset | ~3.5M transitions | From ~3,500 winner replays |
| Train/val split | 90/10 | |
| Loss | Factored cross-entropy | Source + Target + Fraction |

### PPO Training
| Parameter | Value | Notes |
|-----------|-------|-------|
| Learning rate | 3e-5 | 10× smaller than BC |
| Discount (γ) | 0.997 | High — long-horizon game |
| GAE λ | 0.95 | Standard |
| Clip ε | 0.2 | Standard PPO clip range |
| Value coef | 0.5 | Weight of value loss |
| Entropy coef | 0.05 | Higher than usual — prevents premature convergence in 4p |
| Noop penalty | 4.0 | Logit bias against doing nothing |
| Idle penalty | 0.02 | Reward penalty for noop with ships available |
| Step penalty | 0.002 | Small per-step cost to encourage decisive play |
| Episodes per update | 16 | Rollouts collected before each PPO update |
| PPO epochs | 4 | Reuse each batch 4 times |
| Mini-batch size | 64 | |
| Win bonus | ±10.0 | Terminal reward scale |
| Rollout device | CPU | Game env doesn't support GPU |
| Update device | MPS/CUDA | GPU-accelerated gradient updates |

### Curriculum
| Parameter | Value | Notes |
|-----------|-------|-------|
| Stages | 8 | Floor → Elite |
| Opponent tiers | 8 | Random(1) → Pilkwang(8) |
| Self-play refresh | Every 20 episodes | Frozen copy of current policy |
| Lagging refresh | Every 2000 episodes | "Old self" for progress tracking |
| 4p fraction | 5% → 50% | Over 10,000 episodes |

### Progressive Horizon
| WR Threshold | Horizon | Notes |
|-------------|---------|-------|
| 0% | 100 steps | Learn basics |
| 50% | 200 steps | Mid-game |
| 60% | 350 steps | Late-game |
| 70% | 500 steps | Full game |

## 13. Training Commands

### Step 1: Parse Replays and Train BC

```bash
# Parse replays and pre-train with behavioral cloning
python -m ppo_gnn.train_bc \
    --mode 4p \
    --replay-dir kaggle_replays \
    --hidden-dim 256 \
    --batch-size 512 \
    --lr 3e-4 \
    --epochs 20 \
    --device mps
```

This produces `ppo_gnn/cache/checkpoint_bc_4p.pt`.

### Step 2: PPO Fine-Tuning

```bash
python -m ppo_gnn.train_ppo \
    --mode mixed \
    --checkpoint ppo_gnn/cache/checkpoint_bc_4p.pt \
    --hidden-dim 256 \
    --mask-sun \
    --separate-critic \
    --progressive-horizon \
    --lr 3e-5 \
    --entropy-coef 0.05 \
    --gamma 0.997 \
    --noop-penalty 4.0 \
    --idle-penalty 0.02 \
    --step-penalty 0.002 \
    --episodes-per-update 16 \
    --device cpu \
    --update-device mps
```

This loads the BC checkpoint, initialises the separate critic backbone from the policy weights, and begins PPO training with curriculum opponents.

## 14. Monitoring Training

The training loop logs key metrics every update:

```
Update: policy=0.13 value=0.13 entropy=3.25 kl=0.99 clip=0.90 | 
wr(50)=81% heur_wr=0.0% avg_r=+1.89 base_wr=0% lag_wr=50% 
stage=floor(tier≤1) horizon=100
```

What to watch:

| Metric | Healthy Range | Meaning |
|--------|--------------|--------|
| `policy` | 0.01–0.3 | Policy loss — lower is better but too low means no learning |
| `value` | 0.1–1.0 | Value prediction error — should decrease over time |
| `entropy` | 2.0–5.0 | Action randomness — too low = collapsed policy, too high = random |
| `kl` | 0.005–0.05 | How much policy changed — too high = unstable updates |
| `clip` | 0.1–0.5 | Fraction of clipped ratios — 0 = no clipping needed, 1 = all clipped |
| `wr(50)` | Rising | Overall win rate (50-game window) |
| `heur_wr` | Rising | Win rate vs heuristic opponents only (drives curriculum advancement) |
| `stage` | Advancing | Current curriculum stage |
| `horizon` | Extending | Current max game steps |

## 15. Key Lessons Learned

### What Worked
1. **BC pre-training is essential** — random PPO takes forever to learn basic fleet launching
2. **Edge features in messages, not just attention** — huge improvement in spatial reasoning
3. **Separate critic backbone** — prevents policy degradation during value updates, especially with self-play
4. **Progressive horizon** — the agent learns basics fast on short games, then transfers to full-length
5. **Dense reward shaping** — production-weighted scoring gives meaningful per-step signal
6. **Feature engineering matters** — going from 10 to 28 features significantly improved learning speed
7. **Noop penalty as logit bias** — more effective than reward shaping alone for preventing idle agents

### What Didn't Work
1. **Shared actor-critic backbone** — policy and value gradients interfere, especially during self-play. The value loss drags policy features around, causing instability
2. **Too much self-play too early** — a weak policy playing itself learns nothing. Start with heuristic opponents
3. **Fixed 500-step horizon from start** — credit assignment over 500 steps is too hard for a new policy
4. **Low entropy coefficient** — the default 0.015 causes premature convergence in 4p games. 0.05 keeps exploration alive
5. **Starting PPO from BC checkpoint when PPO checkpoint exists** — you lose all the RL-specific learning

## 16. Code Structure

```
ppo_gnn/
├── gnn_policy.py        # GNN architecture (GAT layers, policy heads, critic)
├── replay_parser.py     # Parse Kaggle replays → graph training data
├── train_bc.py          # Phase 1: Behavioral Cloning training loop
├── train_ppo.py         # Phase 2: PPO training with curriculum
├── sun_geometry.py      # Sun intersection computation for edge features
├── test_gnn.py          # Unit tests for the GNN
└── cache/               # Checkpoints, datasets, logs
    ├── checkpoint_bc_4p.pt
    └── checkpoint_ppo_mixed_latest.pt

submission_gnn/
└── main.py              # Kaggle submission agent (self-contained)
```

## 17. Summary

Building a competitive Orbit Wars agent requires solving several interlinked challenges:

1. **Representation**: Model the game as a graph with rich node features (28-dim) and edge features (6-dim) that encode ownership, military balance, spatial relationships, and sun geometry

2. **Architecture**: Use GAT with edge-conditioned messages so the network reasons about *"what is this neighbor AND how far away is it?"* simultaneously. Separate the actor and critic backbones to prevent gradient interference

3. **Action space**: Factor the enormous action space into three sequential decisions (source → target → fraction) with appropriate masking

4. **Training pipeline**: 
   - **BC** teaches basic competence from expert replays
   - **PPO** refines through self-play and curriculum opponents
   - **Progressive horizon** builds from short to full-length games
   - **Curriculum stages** introduce harder opponents gradually

5. **Reward engineering**: Dense per-step rewards (production-weighted scoring), noop/idle penalties, and careful terminal reward design with survival bonuses and horizon-cutoff scoring

The result is a 1.24M parameter agent that learns to play both 2-player and 4-player Orbit Wars from a single network, adapting its strategy based on the game mode, phase, and opponent behaviour.